# ABSA Hotel Santika - Data Splitting + Fine-Tuning IndoBERT (Kaggle Ready)

Notebook ini menggabungkan **dua tahap** dalam satu alur yang reproducible:

1. **Data Splitting** - multi-label stratified split (train/validation/test = 80/10/10) dari
   `dataset_absa_labeled.csv` (15.097 review, 7 aspek, 4 kelas: `none/positif/negatif/netral`).
2. **Fine-Tuning IndoBERT** - arsitektur **multi-head** (1 encoder IndoBERT + 7 classifier head),
   dengan **hyperparameter search** (curated / random / grid) untuk mengejar macro-F1 tertinggi.

### Kondisi dataset (hasil merge -> preprocessing -> labeling)
- 7 aspek: `Kenyamanan, Kebersihan, Pelayanan, Harga, Lokasi, Fasilitas, Makanan`.
- Distribusi sangat **imbalanced** (mis. Harga hanya ~1.031 berlabel; kelas `netral` sangat minoritas).
- Karena itu: **stratified split per (aspek, label)**, **class weighting**, **label smoothing**,
  dan metrik evaluasi yang relevan untuk ABSA (`non_none_macro_f1`, `aspect_detection_f1`,
  `false_aspect_rate`).
- IndoBERT yang dipakai **uncased** -> teks di-*lowercase* sebelum tokenisasi (preprocessing minimal,
  tanpa stemming/stopword removal agar konteks sentimen tidak hilang).

### Cara pakai di Kaggle
1. Upload `dataset_absa_labeled.csv` sebagai Kaggle Dataset (resolver di bawah mencari otomatis).
2. Settings: Accelerator = **GPU T4 x2** (dual GPU didukung via DataParallel), Internet = **ON** (untuk unduh model HF).
   - Set `USE_MULTI_GPU=True` (default) agar kedua GPU dipakai. Dengan T4 x2, `batch_size=32` aman karena beban terbagi ke 2 GPU.
3. Run All. Hasil split + model terbaik + metrik disimpan ke `/kaggle/working`.


## 1. Setup & Dependencies

In [1]:
import sys, subprocess
def pipi(p): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *p])
# PENTING: jangan menurunkan numpy. Kaggle berbasis numpy 2.x.
# transformers>=4.46 kompatibel dengan numpy 2.x. sentencepiece dibutuhkan tokenizer indobert-lite.
pipi(['transformers>=4.46,<5', 'accelerate>=0.34', 'sentencepiece', 'iterative-stratification>=0.1.7'])
import numpy, transformers
print('done. numpy', numpy.__version__, '| transformers', transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

done. numpy 2.4.6 | transformers 4.57.6


In [2]:
import os, json, random, gc, itertools, time, traceback
from pathlib import Path

import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, BertTokenizer, AutoModel,
                          get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup)
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# === Konfigurasi multi-GPU (Kaggle T4 x2) ===
# USE_MULTI_GPU=True -> pakai nn.DataParallel agar batch dibagi ke semua GPU yang tersedia.
# USE_AMP=True       -> mixed precision (fp16) untuk hemat memori & lebih cepat di T4.
USE_MULTI_GPU = True
USE_AMP = True

N_GPU = torch.cuda.device_count() if DEVICE == 'cuda' else 0
print('Device:', DEVICE, '| jumlah GPU:', N_GPU)
for gi in range(N_GPU):
    print(f'  GPU{gi}:', torch.cuda.get_device_name(gi))
if USE_MULTI_GPU and N_GPU > 1:
    print(f'Multi-GPU AKTIF: DataParallel akan memakai {N_GPU} GPU. '
          f'batch_size efektif dibagi rata per GPU.')
else:
    print('Single-GPU mode.')

Device: cuda | jumlah GPU: 2
  GPU0: Tesla T4
  GPU1: Tesla T4
Multi-GPU AKTIF: DataParallel akan memakai 2 GPU. batch_size efektif dibagi rata per GPU.


## 2. Konfigurasi Global (aspek, label, path)

In [3]:
ASPECTS = ['Kenyamanan', 'Kebersihan', 'Pelayanan', 'Harga', 'Lokasi', 'Fasilitas', 'Makanan']
LABEL2ID = {'none': 0, 'positif': 1, 'negatif': 2, 'netral': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABEL_NAMES = ['none', 'positif', 'negatif', 'netral']
NUM_CLASSES = 4

TEXT_COL = 'Text_Review'
ID_COL = 'ID_Review'

SEED = 42
SPLIT_RATIOS = {'train': 0.8, 'validation': 0.1, 'test': 0.1}

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
WORKING.mkdir(parents=True, exist_ok=True)
SPLIT_OUT = WORKING / 'absa_santika_split'
SPLIT_OUT.mkdir(parents=True, exist_ok=True)


def resolve_dataset(filename='dataset_absa_labeled.csv'):
    # Cari di Kaggle Input, fallback ke direktori lokal.
    if KAGGLE_INPUT.exists():
        hits = sorted(KAGGLE_INPUT.rglob(filename), key=lambda x: str(x).lower())
        if hits:
            return hits[0]
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        hits = sorted(root.rglob(filename), key=lambda x: str(x).lower())
        if hits:
            return hits[0]
    raise FileNotFoundError(f'{filename} tidak ditemukan. Upload dataset ke Kaggle Input.')


DATA_PATH = resolve_dataset()
print('Dataset:', DATA_PATH)

Dataset: /kaggle/input/datasets/vince0014/dataset-label/dataset_absa_labeled.csv


## 3. Load & Normalisasi Label

In [4]:
def nlab(v):
    v = str(v).strip().lower()
    return v if v in LABEL2ID else 'none'


df = pd.read_csv(DATA_PATH, encoding='utf-8-sig', dtype=str).fillna('')
print('Shape awal:', df.shape)

# Validasi kolom wajib
missing = [c for c in [TEXT_COL] + ASPECTS if c not in df.columns]
if missing:
    raise ValueError(f'Kolom hilang: {missing}. Tersedia: {list(df.columns)}')

# Normalisasi label aspek
for a in ASPECTS:
    df[a] = df[a].map(nlab)

# Buang baris teks kosong
df = df[df[TEXT_COL].astype(str).str.strip() != ''].reset_index(drop=True)

# Kolom teks lowercase untuk IndoBERT uncased (preprocessing minimal)
df['text_model'] = df[TEXT_COL].astype(str).str.strip().str.lower()

# Pastikan ID unik untuk audit leakage
if ID_COL not in df.columns:
    df[ID_COL] = ['R' + str(i) for i in range(len(df))]
df[ID_COL] = df[ID_COL].astype(str)
if df[ID_COL].duplicated().any():
    df[ID_COL] = ['R' + str(i) for i in range(len(df))]

print('Shape setelah cleaning:', df.shape)

# Ringkas distribusi
rows = []
for a in ASPECTS:
    vc = df[a].value_counts().to_dict()
    rows.append({'aspect': a, **{k: vc.get(k, 0) for k in LABEL_NAMES},
                 'berlabel': int((df[a] != 'none').sum())})
display(pd.DataFrame(rows))
n_labeled = int((df[ASPECTS].ne('none').any(axis=1)).sum())
print(f'Review dengan >=1 aspek: {n_labeled} | tanpa aspek: {len(df) - n_labeled}')

Shape awal: (15097, 19)
Shape setelah cleaning: (15097, 20)


,aspect,none,positif,negatif,netral,berlabel
0,Kenyamanan,9234,3985,1619,259,5863
1,Kebersihan,10852,3556,616,73,4245
2,Pelayanan,8790,5070,875,362,6307
3,Harga,14066,639,254,138,1031
4,Lokasi,10431,4192,282,192,4666
5,Fasilitas,11482,1465,1589,561,3615
6,Makanan,9741,3875,915,566,5356


Review dengan >=1 aspek: 13594 | tanpa aspek: 1503


## 4. Multi-Label Stratified Split (80/10/10)

Karena setiap review punya **7 label aspek sekaligus** dan distribusinya sangat timpang,
splitting acak biasa berisiko membuat kelas minoritas (mis. `Harga=netral`) tidak terwakili di
val/test. Kita pakai **iterative multi-label stratification** (Sechidis et al. 2011) lewat
`MultilabelStratifiedKFold`. Tiap pasangan (aspek, label) dijadikan indikator one-hot
sehingga proporsinya terjaga di setiap split.

Fallback: jika library tidak tersedia, dipakai greedy iterative stratified split manual.


In [5]:
def build_indicator_matrix(frame):
    # One-hot indikator untuk tiap (aspek, label) -> shape (n, 7*4)
    cols = []
    for a in ASPECTS:
        for lab in LABEL_NAMES:
            cols.append((frame[a] == lab).astype(int).values)
    return np.stack(cols, axis=1)


Y_ind = build_indicator_matrix(df)
idx_all = np.arange(len(df))


def stratified_split(frame, Y, ratios, seed=SEED):
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
        # Tahap 1: pisahkan test (10%)
        n_splits_1 = int(round(1 / ratios['test']))
        mskf1 = MultilabelStratifiedKFold(n_splits=n_splits_1, shuffle=True, random_state=seed)
        rest_idx, test_idx = next(mskf1.split(np.zeros(len(frame)), Y))
        # Tahap 2: dari sisa, pisahkan validation
        val_frac_rest = ratios['validation'] / (ratios['train'] + ratios['validation'])
        n_splits_2 = int(round(1 / val_frac_rest))
        mskf2 = MultilabelStratifiedKFold(n_splits=n_splits_2, shuffle=True, random_state=seed)
        sub_train, sub_val = next(mskf2.split(np.zeros(len(rest_idx)), Y[rest_idx]))
        train_idx = rest_idx[sub_train]
        val_idx = rest_idx[sub_val]
        method = 'MultilabelStratifiedKFold (iterative-stratification)'
    except Exception as e:
        print('[WARN] iterative-stratification gagal, fallback greedy:', e)
        train_idx, val_idx, test_idx = greedy_stratified(Y, ratios, seed)
        method = 'greedy iterative multi-label stratified split'
    return np.sort(train_idx), np.sort(val_idx), np.sort(test_idx), method


def greedy_stratified(Y, ratios, seed=SEED):
    rng = np.random.RandomState(seed)
    n, k = Y.shape
    targets = {'train': ratios['train'], 'validation': ratios['validation'], 'test': ratios['test']}
    desired = {s: targets[s] * Y.sum(axis=0) for s in targets}
    counts = {s: np.zeros(k) for s in targets}
    sizes = {s: 0 for s in targets}
    size_cap = {s: targets[s] * n for s in targets}
    order = np.argsort(-Y.sum(axis=1))
    order = rng.permutation(order)
    assign = {}
    for i in order:
        row = Y[i]
        best_s, best_score = None, None
        for s in targets:
            if sizes[s] >= size_cap[s] + 1:
                continue
            score = ((desired[s] - counts[s]) * row).sum()
            tie = -sizes[s]
            cand = (score, tie)
            if best_score is None or cand > best_score:
                best_score, best_s = cand, s
        if best_s is None:
            best_s = min(sizes, key=lambda s: sizes[s] / size_cap[s])
        assign[i] = best_s
        counts[best_s] += row
        sizes[best_s] += 1
    train_idx = np.array([i for i, s in assign.items() if s == 'train'])
    val_idx = np.array([i for i, s in assign.items() if s == 'validation'])
    test_idx = np.array([i for i, s in assign.items() if s == 'test'])
    return train_idx, val_idx, test_idx


train_idx, val_idx, test_idx, split_method = stratified_split(df, Y_ind, SPLIT_RATIOS, SEED)
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
print('Metode split:', split_method)
print('train/val/test:', len(train_df), len(val_df), len(test_df))

Metode split: MultilabelStratifiedKFold (iterative-stratification)
train/val/test: 12076 1510 1511


In [6]:
# Audit data leakage (tidak boleh ada ID yang sama antar split)
ti, vi, tei = set(train_df[ID_COL]), set(val_df[ID_COL]), set(test_df[ID_COL])
overlaps = {'train_val': len(ti & vi), 'train_test': len(ti & tei), 'val_test': len(vi & tei)}
print('Overlap antar split:', overlaps)
assert sum(overlaps.values()) == 0, 'Data leakage terdeteksi!'

# Audit kualitas stratifikasi: bandingkan proporsi tiap (aspek,label) di full vs tiap split
def proportions(frame):
    p = {}
    for a in ASPECTS:
        for lab in LABEL_NAMES:
            p[(a, lab)] = (frame[a] == lab).mean()
    return p

p_full = proportions(df)
diffs = []
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    p = proportions(frame)
    for key in p_full:
        diffs.append({'split': name, 'aspect': key[0], 'label': key[1],
                      'full_pct': p_full[key], 'split_pct': p[key],
                      'abs_diff': abs(p_full[key] - p[key])})
diff_df = pd.DataFrame(diffs)
print('Max abs diff proporsi (aspek,label):', round(diff_df['abs_diff'].max(), 5))
print('Mean abs diff proporsi (aspek,label):', round(diff_df['abs_diff'].mean(), 5))
display(diff_df.sort_values('abs_diff', ascending=False).head(10))

Overlap antar split: {'train_val': 0, 'train_test': 0, 'val_test': 0}
Max abs diff proporsi (aspek,label): 0.00086
Mean abs diff proporsi (aspek,label): 0.00017


,split,aspect,label,full_pct,split_pct,abs_diff
52,validation,Makanan,none,0.645228,0.644371,0.000857
72,test,Lokasi,none,0.690932,0.690271,0.000661
80,test,Makanan,none,0.645228,0.644606,0.000621
60,test,Kebersihan,none,0.718818,0.719391,0.000573
75,test,Lokasi,netral,0.012718,0.013236,0.000519
44,validation,Lokasi,none,0.690932,0.691391,0.000459
62,test,Kebersihan,negatif,0.040803,0.040371,0.000432
28,validation,Kenyamanan,none,0.611645,0.611258,0.000386
48,validation,Fasilitas,none,0.760548,0.760927,0.000379
49,validation,Fasilitas,positif,0.097039,0.096689,0.000350


In [7]:
# Simpan split + manifest ke /kaggle/working
SAVE_COLS = [c for c in [ID_COL, 'Platform', 'Nama_Hotel', 'Review_Date', TEXT_COL, 'text_model'] + ASPECTS if c in df.columns]
train_df[SAVE_COLS].to_csv(SPLIT_OUT / 'train.csv', index=False, encoding='utf-8-sig')
val_df[SAVE_COLS].to_csv(SPLIT_OUT / 'validation.csv', index=False, encoding='utf-8-sig')
test_df[SAVE_COLS].to_csv(SPLIT_OUT / 'test.csv', index=False, encoding='utf-8-sig')

manifest = {
    'dataset_path': str(DATA_PATH),
    'seed': SEED,
    'split_ratios': SPLIT_RATIOS,
    'actual_sizes': {'train': len(train_df), 'validation': len(val_df), 'test': len(test_df)},
    'n_rows': len(df),
    'aspects': ASPECTS,
    'labels': LABEL_NAMES,
    'method': split_method,
    'max_abs_pct_diff_aspect_label': float(diff_df['abs_diff'].max()),
    'mean_abs_pct_diff_aspect_label': float(diff_df['abs_diff'].mean()),
    'overlaps': overlaps,
}
with open(SPLIT_OUT / 'split_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print('Split tersimpan di:', SPLIT_OUT)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

Split tersimpan di: /kaggle/working/absa_santika_split
{
  "dataset_path": "/kaggle/input/datasets/vince0014/dataset-label/dataset_absa_labeled.csv",
  "seed": 42,
  "split_ratios": {
    "train": 0.8,
    "validation": 0.1,
    "test": 0.1
  },
  "actual_sizes": {
    "train": 12076,
    "validation": 1510,
    "test": 1511
  },
  "n_rows": 15097,
  "aspects": [
    "Kenyamanan",
    "Kebersihan",
    "Pelayanan",
    "Harga",
    "Lokasi",
    "Fasilitas",
    "Makanan"
  ],
  "labels": [
    "none",
    "positif",
    "negatif",
    "netral"
  ],
  "method": "MultilabelStratifiedKFold (iterative-stratification)",
  "max_abs_pct_diff_aspect_label": 0.0008566677209234097,
  "mean_abs_pct_diff_aspect_label": 0.00016532831052204827,
  "overlaps": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  }
}


## 5. Targets & Class Weights

In [8]:
def targets(frame):
    Y = np.zeros((len(frame), len(ASPECTS)), dtype=np.int64)
    for i, a in enumerate(ASPECTS):
        Y[:, i] = frame[a].map(nlab).map(LABEL2ID).astype(np.int64).values
    return Y


y_train, y_val, y_test = targets(train_df), targets(val_df), targets(test_df)

# Class weighting: sqrt inverse-frequency + clipping (lebih stabil daripada inverse penuh)
CLASS_WEIGHT_MODE = 'sqrt_capped'
CLASS_WEIGHT_MAX = 4.0
CLASS_WEIGHT_MIN = 0.25


def cls_weights(y, mode=CLASS_WEIGHT_MODE):
    W = np.ones((len(ASPECTS), NUM_CLASSES), dtype=np.float32)
    for i in range(len(ASPECTS)):
        vals, cnts = np.unique(y[:, i], return_counts=True)
        freq = np.zeros(NUM_CLASSES, dtype=np.float32)
        freq[vals] = cnts
        freq[freq == 0] = 1.0
        inv = freq.sum() / (NUM_CLASSES * freq)
        if mode == 'sqrt_capped':
            w = np.sqrt(inv); w = np.clip(w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX); w = w / w.mean()
        elif mode == 'inverse_capped':
            w = np.clip(inv, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX); w = w / w.mean()
        else:
            w = np.ones(NUM_CLASSES, dtype=np.float32)
        W[i] = w
    return torch.tensor(W, dtype=torch.float32).to(DEVICE)


CLASS_W = cls_weights(y_train)
display(pd.DataFrame(CLASS_W.detach().cpu().numpy(), index=ASPECTS, columns=LABEL_NAMES).round(3))

,none,positif,negatif,netral
Kenyamanan,0.367,0.559,0.878,2.195
Kebersihan,0.291,0.509,1.223,1.977
Pelayanan,0.384,0.506,1.218,1.892
Harga,0.192,0.901,1.425,1.482
Lokasi,0.261,0.412,1.588,1.738
Fasilitas,0.363,1.017,0.977,1.643
Makanan,0.400,0.634,1.306,1.660


## 6. Dataset, Model Multi-Head, Loss

In [9]:
class DS(Dataset):
    def __init__(self, texts, Y, tok, ml):
        self.t = list(texts); self.Y = Y; self.tok = tok; self.ml = ml
    def __len__(self):
        return len(self.t)
    def __getitem__(self, i):
        e = self.tok(self.t[i], truncation=True, max_length=self.ml,
                     padding='max_length', return_tensors='pt')
        it = {k: v.squeeze(0) for k, v in e.items()}
        it['labels'] = torch.tensor(self.Y[i])
        return it


class MultiHead(nn.Module):
    """1 encoder IndoBERT + 7 classifier head (satu head per aspek, 4 kelas)."""
    def __init__(self, name, na, nc, dropout=0.1, freeze=0):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(h, nc) for _ in range(na)])
        if freeze > 0:
            for p in self.enc.embeddings.parameters():
                p.requires_grad = False
            layers = getattr(self.enc.encoder, 'layer', [])
            for l in layers[:freeze]:
                for p in l.parameters():
                    p.requires_grad = False
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # autocast di DALAM forward: thread replika DataParallel tidak mewarisi
        # autocast dari thread utama, jadi harus di sini agar AMP bekerja multi-GPU.
        with torch.amp.autocast('cuda', enabled=(USE_AMP and input_ids.is_cuda)):
            o = self.enc(input_ids=input_ids, attention_mask=attention_mask)
            cls = self.drop(o.last_hidden_state[:, 0])
            logits = torch.stack([hd(cls) for hd in self.heads], dim=1)
        # Keluar dari autocast & paksa float32: CrossEntropyLoss dengan weight float32
        # tidak menerima input Half. Loss memang lebih stabil dihitung di fp32.
        return logits.float()


def make_loss(cw, smoothing):
    if cw:
        outs = [nn.CrossEntropyLoss(weight=CLASS_W[i], label_smoothing=smoothing) for i in range(len(ASPECTS))]
    else:
        outs = [nn.CrossEntropyLoss(label_smoothing=smoothing) for _ in range(len(ASPECTS))]
    def fn(logits, labels):
        return sum(outs[i](logits[:, i, :], labels[:, i]) for i in range(len(ASPECTS))) / len(ASPECTS)
    return fn

## 7. Metrik ABSA & Evaluasi

In [10]:
def aspect_metrics(y_true, y_pred, aspect_idx):
    yt = y_true[:, aspect_idx]; yp = y_pred[:, aspect_idx]
    true_none_mask = yt == LABEL2ID['none']
    present_mask = yt != LABEL2ID['none']
    macro_f1 = f1_score(yt, yp, labels=[0, 1, 2, 3], average='macro', zero_division=0)
    weighted_f1 = f1_score(yt, yp, labels=[0, 1, 2, 3], average='weighted', zero_division=0)
    acc = accuracy_score(yt, yp)
    non_none_macro_f1 = f1_score(yt, yp, labels=[1, 2, 3], average='macro', zero_division=0)
    aspect_detection_f1 = f1_score((yt != 0).astype(int), (yp != 0).astype(int), zero_division=0)
    false_aspect_rate = float((yp[true_none_mask] != 0).mean()) if true_none_mask.any() else 0.0
    sentiment_macro_f1_present = (f1_score(yt[present_mask], yp[present_mask], labels=[1, 2, 3],
                                  average='macro', zero_division=0) if present_mask.any() else 0.0)
    return {'macro_f1': float(macro_f1), 'weighted_f1': float(weighted_f1), 'acc': float(acc),
            'non_none_macro_f1': float(non_none_macro_f1), 'aspect_detection_f1': float(aspect_detection_f1),
            'false_aspect_rate': float(false_aspect_rate),
            'sentiment_macro_f1_present': float(sentiment_macro_f1_present)}


@torch.no_grad()
def evaluate(model, loader, ret=False):
    model.eval()
    logits_all, labels_all = [], []
    for batch in loader:
        labels = batch.pop('labels').to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'token_type_ids']}
        logits_all.append(model(**inputs).cpu())
        labels_all.append(labels.cpu())
    L = torch.cat(logits_all); Y = torch.cat(labels_all).numpy(); P = L.argmax(-1).numpy()
    per = {a: aspect_metrics(Y, P, i) for i, a in enumerate(ASPECTS)}
    r = {m: float(np.mean([per[a][m] for a in ASPECTS])) for m in
         ['macro_f1', 'weighted_f1', 'acc', 'non_none_macro_f1', 'aspect_detection_f1',
          'false_aspect_rate', 'sentiment_macro_f1_present']}
    r['per_aspect'] = per
    return (r, P, Y) if ret else r

## 8. Training Loop + Early Stopping + AMP

In [11]:
def build_opt(model, cfg):
    if cfg.get('optimizer', 'adamw') == 'adafactor':
        from transformers.optimization import Adafactor
        return Adafactor(model.parameters(), lr=cfg['lr'], scale_parameter=False,
                         relative_step=False, warmup_init=False)
    return AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('weight_decay', 0.01))


def build_sched(opt, cfg, total):
    w = int(cfg.get('warmup_ratio', 0.1) * total)
    if cfg.get('scheduler', 'linear') == 'cosine':
        return get_cosine_schedule_with_warmup(opt, w, total)
    return get_linear_schedule_with_warmup(opt, w, total)


def train_cfg(cfg, verbose=True):
    set_seed(cfg.get('seed', 42))
    name = cfg['model_name']
    # indobert-lite = arsitektur ALBERT tapi vocab WordPiece -> pakai BertTokenizer.
    tok = (BertTokenizer if 'lite' in name.lower() else AutoTokenizer).from_pretrained(name)
    trL = DataLoader(DS(train_df['text_model'].tolist(), y_train, tok, cfg['max_len']),
                     batch_size=cfg['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    vaL = DataLoader(DS(val_df['text_model'].tolist(), y_val, tok, cfg['max_len']),
                     batch_size=cfg['batch_size'], num_workers=2, pin_memory=True)
    model = MultiHead(name, len(ASPECTS), NUM_CLASSES, cfg.get('dropout', 0.1), cfg.get('freeze_layers', 0)).to(DEVICE)
    if USE_MULTI_GPU and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    loss_fn = make_loss(cfg.get('class_weight', False), cfg.get('label_smoothing', 0.0))
    opt = build_opt(model, cfg)
    accum = cfg.get('grad_accum', 1)
    total = (len(trL) // accum) * cfg['max_epochs']
    sched = build_sched(opt, cfg, max(total, 1))
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE == 'cuda'))
    best = -1; best_state = None; bad = 0
    patience = cfg.get('patience', 2); hist = []
    for ep in range(cfg['max_epochs']):
        model.train(); opt.zero_grad()
        for step, b in enumerate(trL):
            lb = b.pop('labels').to(DEVICE)
            b = {k: v.to(DEVICE) for k, v in b.items() if k in ['input_ids', 'attention_mask', 'token_type_ids']}
            # autocast sudah ditangani di dalam MultiHead.forward (kompatibel DataParallel)
            loss = loss_fn(model(**b), lb) / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        r = evaluate(model, vaL); hist.append(r['macro_f1'])
        if verbose:
            print(f"   ep{ep+1}/{cfg['max_epochs']} valF1={r['macro_f1']:.4f} "
                  f"nonNoneF1={r['non_none_macro_f1']:.4f} aspDet={r['aspect_detection_f1']:.4f}")
        if r['macro_f1'] > best + 1e-4:
            best = r['macro_f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f'   early stop @ epoch {ep+1}')
                break
    if best_state:
        model.load_state_dict(best_state)
    return model, tok, best, hist, (ep + 1)

## 9. Ruang Pencarian Hyperparameter

Atur `SEARCH_MODE`:
- `'curated'` : 10 konfigurasi pilihan (cepat, direkomendasikan untuk run utama).
- `'random'`  : sampel acak dari ruang besar sebanyak `N_RANDOM` (Bergstra & Bengio 2012).
- `'grid'`    : full cartesian dari subset key (HATI-HATI: bisa sangat banyak & lama).

Model kandidat IndoBERT uncased:
- `indobenchmark/indobert-base-p1` (P1) - baseline kuat.
- `indolem/indobert-base-uncased` (LEM) - alternatif uncased.
- `indobenchmark/indobert-lite-base-p1` (LITE) - ringan & cepat.


In [12]:
P1 = 'indobenchmark/indobert-base-p1'
LEM = 'indolem/indobert-base-uncased'
LITE = 'indobenchmark/indobert-lite-base-p1'

SPACE = dict(
    model_name=[P1, LEM, LITE],
    lr=[1e-5, 2e-5, 3e-5, 5e-5],
    batch_size=[8, 16, 32],
    max_epochs=[6, 8, 10],
    max_len=[128, 160, 192],
    warmup_ratio=[0.06, 0.1],
    weight_decay=[0.01, 0.1],
    dropout=[0.1, 0.2, 0.3],
    scheduler=['linear', 'cosine'],
    optimizer=['adamw'],
    label_smoothing=[0.0, 0.05, 0.1],
    freeze_layers=[0, 6],
    class_weight=[False, True],
    seed=[42, 7, 123],
)

CURATED = [
    dict(name='c1_p1_base_128',    model_name=P1, lr=2e-5, batch_size=16, max_epochs=8,  max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0,  freeze_layers=0, class_weight=False, patience=2, seed=42),
    dict(name='c2_p1_lr3_cw',      model_name=P1, lr=3e-5, batch_size=16, max_epochs=10, max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.2, scheduler='linear', optimizer='adamw', label_smoothing=0.05, freeze_layers=0, class_weight=True,  patience=3, seed=42),
    dict(name='c3_p1_cw_160',      model_name=P1, lr=3e-5, batch_size=16, max_epochs=10, max_len=160, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.2, scheduler='linear', optimizer='adamw', label_smoothing=0.05, freeze_layers=0, class_weight=True,  patience=3, seed=42),
    dict(name='c4_p1_cw_192_bs8',  model_name=P1, lr=2e-5, batch_size=8,  max_epochs=10, max_len=192, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.2, scheduler='cosine', optimizer='adamw', label_smoothing=0.05, freeze_layers=0, class_weight=True,  grad_accum=2, patience=3, seed=42),
    dict(name='c5_p1_cos_160',     model_name=P1, lr=2e-5, batch_size=16, max_epochs=8,  max_len=160, warmup_ratio=0.06, weight_decay=0.01, dropout=0.1, scheduler='cosine', optimizer='adamw', label_smoothing=0.0,  freeze_layers=0, class_weight=False, patience=2, seed=42),
    dict(name='c6_p1_freeze6',     model_name=P1, lr=3e-5, batch_size=32, max_epochs=10, max_len=160, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0,  freeze_layers=6, class_weight=False, patience=3, seed=42),
    dict(name='c7_p1_cw_ls01',     model_name=P1, lr=3e-5, batch_size=16, max_epochs=10, max_len=160, warmup_ratio=0.1,  weight_decay=0.1,  dropout=0.3, scheduler='cosine', optimizer='adamw', label_smoothing=0.1,  freeze_layers=0, class_weight=True,  patience=3, seed=42),
    dict(name='c8_lem_cw_160',     model_name=LEM, lr=3e-5, batch_size=16, max_epochs=10, max_len=160, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.2, scheduler='cosine', optimizer='adamw', label_smoothing=0.05, freeze_layers=0, class_weight=True,  patience=3, seed=42),
    dict(name='c9_lem_base_128',   model_name=LEM, lr=2e-5, batch_size=16, max_epochs=8,  max_len=128, warmup_ratio=0.1,  weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0,  freeze_layers=0, class_weight=False, patience=2, seed=42),
    dict(name='c10_lite_fast_160', model_name=LITE, lr=3e-5, batch_size=32, max_epochs=10, max_len=160, warmup_ratio=0.1, weight_decay=0.01, dropout=0.1, scheduler='linear', optimizer='adamw', label_smoothing=0.0,  freeze_layers=0, class_weight=False, patience=3, seed=42),
]

In [13]:
SEARCH_MODE = 'curated'   # 'curated' | 'random' | 'grid'
N_RANDOM = 8               # dipakai jika SEARCH_MODE='random'


def sample_random(n, seed=42):
    rng = random.Random(seed); out = []
    for j in range(n):
        cfg = {k: rng.choice(v) for k, v in SPACE.items()}
        cfg['patience'] = 2; cfg['name'] = f'rand_{j+1}'
        out.append(cfg)
    return out


def build_grid(keys):
    base = dict(model_name=P1, max_len=128, warmup_ratio=0.1, weight_decay=0.01, dropout=0.1,
                scheduler='linear', optimizer='adamw', label_smoothing=0.0, freeze_layers=0,
                class_weight=False, seed=42, patience=2)
    vals = [SPACE[k] for k in keys]; out = []
    for combo in itertools.product(*vals):
        c = dict(base); c.update(dict(zip(keys, combo)))
        c['name'] = 'grid_' + '_'.join(f'{k}{c[k]}' for k in keys)
        out.append(c)
    return out


if SEARCH_MODE == 'curated':
    EXPERIMENTS = CURATED
elif SEARCH_MODE == 'random':
    EXPERIMENTS = sample_random(N_RANDOM)
else:
    EXPERIMENTS = build_grid(['lr', 'batch_size', 'max_epochs'])
print('Total eksperimen:', len(EXPERIMENTS))

Total eksperimen: 10


## 10. Jalankan Eksperimen

In [14]:
def is_oom_error(err):
    msg = str(err).lower()
    return any(t in msg for t in ['out of memory', 'cublas_status_alloc_failed', 'cuda out of memory'])


results = []; best = None; t0 = time.time()
for cfg in EXPERIMENTS:
    print('=' * 64); print('RUN:', cfg['name'])
    try:
        model, tok, vf1, hist, ran_ep = train_cfg(cfg)
    except RuntimeError as e:
        if is_oom_error(e):
            print('  SKIP OOM:', str(e)[:180])
            torch.cuda.empty_cache(); gc.collect(); continue
        print(traceback.format_exc()); raise
    row = {k: cfg.get(k) for k in ['name', 'model_name', 'lr', 'batch_size', 'max_epochs', 'max_len',
           'warmup_ratio', 'weight_decay', 'dropout', 'scheduler', 'optimizer', 'label_smoothing',
           'freeze_layers', 'class_weight', 'seed']}
    row['epochs_ran'] = ran_ep; row['val_macro_f1'] = vf1
    results.append(row)
    print(f'  >> val_macroF1={vf1:.4f} (ran {ran_ep} epoch)')
    if best is None or vf1 > best['vf1']:
        if best is not None:
            del best['model']; gc.collect(); torch.cuda.empty_cache()
        best = {'cfg': cfg, 'vf1': vf1, 'model': model, 'tok': tok}
    else:
        del model; gc.collect(); torch.cuda.empty_cache()

print(f'Total waktu: {(time.time() - t0) / 60:.1f} menit')
if not results or best is None:
    raise RuntimeError('Tidak ada eksperimen yang berhasil. Cek path data / koneksi HF / GPU.')
res_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
res_df.to_csv(WORKING / 'hyperparameter_search_results.csv', index=False)
display(res_df)

RUN: c1_p1_base_128


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


pytorch_model.bin:   0%|          | 0.00/46.7M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.7M [00:00<?, ?B/s]

   ep1/10 valF1=0.5202 nonNoneF1=0.3798 aspDet=0.7798
   ep2/10 valF1=0.6034 nonNoneF1=0.4850 aspDet=0.8749
   ep3/10 valF1=0.6492 nonNoneF1=0.5450 aspDet=0.8994
   ep4/10 valF1=0.6538 nonNoneF1=0.5496 aspDet=0.9147
   ep5/10 valF1=0.6790 nonNoneF1=0.5842 aspDet=0.8937
   ep6/10 valF1=0.6829 nonNoneF1=0.5896 aspDet=0.9029
   ep7/10 valF1=0.6856 nonNoneF1=0.5932 aspDet=0.9051
   ep8/10 valF1=0.6882 nonNoneF1=0.5967 aspDet=0.9028
   ep9/10 valF1=0.6939 nonNoneF1=0.6042 aspDet=0.8976
   ep10/10 valF1=0.6941 nonNoneF1=0.6043 aspDet=0.8999
  >> val_macroF1=0.6941 (ran 10 epoch)
Total waktu: 242.5 menit


,name,model_name,lr,batch_size,max_epochs,max_len,warmup_ratio,weight_decay,dropout,scheduler,optimizer,label_smoothing,freeze_layers,class_weight,seed,epochs_ran,val_macro_f1
0,c4_p1_cw_192_bs8,indobenchmark/indobert-base-p1,0.00002,8,10,192,0.10,0.01,0.2,cosine,adamw,0.05,0,True,42,10,0.747731
1,c8_lem_cw_160,indolem/indobert-base-uncased,0.00003,16,10,160,0.10,0.01,0.2,cosine,adamw,0.05,0,True,42,10,0.746096
2,c7_p1_cw_ls01,indobenchmark/indobert-base-p1,0.00003,16,10,160,0.10,0.10,0.3,cosine,adamw,0.10,0,True,42,10,0.743644
3,c3_p1_cw_160,indobenchmark/indobert-base-p1,0.00003,16,10,160,0.10,0.01,0.2,linear,adamw,0.05,0,True,42,10,0.742541
4,c2_p1_lr3_cw,indobenchmark/indobert-base-p1,0.00003,16,10,128,0.10,0.01,0.2,linear,adamw,0.05,0,True,42,10,0.733330
5,c6_p1_freeze6,indobenchmark/indobert-base-p1,0.00003,32,10,160,0.10,0.01,0.1,linear,adamw,0.00,6,False,42,10,0.728005
6,c5_p1_cos_160,indobenchmark/indobert-base-p1,0.00002,16,8,160,0.06,0.01,0.1,cosine,adamw,0.00,0,False,42,8,0.722928
7,c1_p1_base_128,indobenchmark/indobert-base-p1,0.00002,16,8,128,0.10,0.01,0.1,linear,adamw,0.00,0,False,42,8,0.720191
8,c10_lite_fast_160,indobenchmark/indobert-lite-base-p1,0.00003,32,10,160,0.10,0.01,0.1,linear,adamw,0.00,0,False,42,10,0.694084
9,c9_lem_base_128,indolem/indobert-base-uncased,0.00002,16,8,128,0.10,0.01,0.1,linear,adamw,0.00,0,False,42,8,0.686583


## 11. Evaluasi Model Terbaik di Test Set

In [15]:
best_cfg = best['cfg']; best_model = best['model']; best_tok = best['tok']
print('Best config:', best_cfg['name'], '| val_macroF1:', round(best['vf1'], 4))

teL = DataLoader(DS(test_df['text_model'].tolist(), y_test, best_tok, best_cfg['max_len']),
                 batch_size=best_cfg['batch_size'])
test_res, P_test, Y_test = evaluate(best_model, teL, ret=True)

print('\n=== TEST METRICS (rata-rata 7 aspek) ===')
for m in ['macro_f1', 'weighted_f1', 'acc', 'non_none_macro_f1', 'aspect_detection_f1',
          'false_aspect_rate', 'sentiment_macro_f1_present']:
    print(f'  {m:28s}: {test_res[m]:.4f}')

per_aspect_rows = [{'aspect': a, **test_res['per_aspect'][a]} for a in ASPECTS]
per_aspect_df = pd.DataFrame(per_aspect_rows).round(4)
display(per_aspect_df)

Best config: c4_p1_cw_192_bs8 | val_macroF1: 0.7477

=== TEST METRICS (rata-rata 7 aspek) ===
  macro_f1                    : 0.7455
  weighted_f1                 : 0.9289
  acc                         : 0.9283
  non_none_macro_f1           : 0.6726
  aspect_detection_f1         : 0.9178
  false_aspect_rate           : 0.0434
  sentiment_macro_f1_present  : 0.7156


,aspect,macro_f1,weighted_f1,acc,non_none_macro_f1,aspect_detection_f1,false_aspect_rate,sentiment_macro_f1_present
0,Kenyamanan,0.7049,0.8693,0.8690,0.6363,0.8641,0.1017,0.6860
1,Kebersihan,0.7700,0.9571,0.9570,0.7013,0.9385,0.0230,0.7609
2,Pelayanan,0.7258,0.9114,0.9107,0.6494,0.9393,0.0568,0.6869
3,Harga,0.7578,0.9772,0.9775,0.6794,0.9010,0.0057,0.7124
4,Lokasi,0.7344,0.9533,0.9550,0.6533,0.9515,0.0278,0.6780
5,Fasilitas,0.7443,0.8989,0.8961,0.6746,0.8635,0.0653,0.7464
6,Makanan,0.7810,0.9355,0.9332,0.7142,0.9668,0.0236,0.7388


In [16]:
# Classification report + confusion matrix per aspek
report_lines = []
for i, a in enumerate(ASPECTS):
    report_lines.append('=' * 60)
    report_lines.append(f'ASPEK: {a}')
    rep = classification_report(Y_test[:, i], P_test[:, i], labels=[0, 1, 2, 3],
                                target_names=LABEL_NAMES, zero_division=0)
    report_lines.append(rep)
    cm = confusion_matrix(Y_test[:, i], P_test[:, i], labels=[0, 1, 2, 3])
    report_lines.append('Confusion matrix (rows=true, cols=pred) [none,positif,negatif,netral]:')
    report_lines.append(str(cm))
full_report = '\n'.join(report_lines)
print(full_report)
with open(WORKING / 'test_classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(full_report)

ASPEK: Kenyamanan
              precision    recall  f1-score   support

        none       0.92      0.90      0.91       924
     positif       0.83      0.87      0.85       399
     negatif       0.76      0.80      0.78       162
      netral       0.29      0.27      0.28        26

    accuracy                           0.87      1511
   macro avg       0.70      0.71      0.70      1511
weighted avg       0.87      0.87      0.87      1511

Confusion matrix (rows=true, cols=pred) [none,positif,negatif,netral]:
[[830  66  27   1]
 [ 44 346   3   6]
 [ 19   3 130  10]
 [  6   2  11   7]]
ASPEK: Kebersihan
              precision    recall  f1-score   support

        none       0.98      0.98      0.98      1087
     positif       0.95      0.94      0.94       356
     negatif       0.77      0.75      0.76        61
      netral       0.38      0.43      0.40         7

    accuracy                           0.96      1511
   macro avg       0.77      0.78      0.77      1511
w

## 12. Simpan Model Terbaik & Artefak

In [17]:
BEST_DIR = WORKING / 'best_absa_indobert'
BEST_DIR.mkdir(parents=True, exist_ok=True)

# Simpan bobot, tokenizer, dan metadata.
# Unwrap DataParallel agar state_dict bersih (tanpa prefix 'module.') untuk inference.
_to_save = best_model.module if isinstance(best_model, nn.DataParallel) else best_model
torch.save(_to_save.state_dict(), BEST_DIR / 'model_state.pt')
best_tok.save_pretrained(BEST_DIR / 'tokenizer')

meta = {
    'best_config': {k: best_cfg.get(k) for k in best_cfg if k != 'model'},
    'val_macro_f1': best['vf1'],
    'test_metrics': {m: test_res[m] for m in ['macro_f1', 'weighted_f1', 'acc',
                     'non_none_macro_f1', 'aspect_detection_f1', 'false_aspect_rate',
                     'sentiment_macro_f1_present']},
    'aspects': ASPECTS,
    'label2id': LABEL2ID,
    'text_col': 'text_model (lowercased Text_Review)',
    'architecture': 'MultiHead: 1 IndoBERT encoder + 7 linear heads (4 kelas)',
}
with open(BEST_DIR / 'best_model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
per_aspect_df.to_csv(WORKING / 'test_per_aspect_metrics.csv', index=False)

print('Artefak tersimpan:')
print('  - Split           :', SPLIT_OUT)
print('  - Search results  :', WORKING / 'hyperparameter_search_results.csv')
print('  - Best model      :', BEST_DIR)
print('  - Test report     :', WORKING / 'test_classification_report.txt')
print(json.dumps(meta, indent=2, ensure_ascii=False))

Artefak tersimpan:
  - Split           : /kaggle/working/absa_santika_split
  - Search results  : /kaggle/working/hyperparameter_search_results.csv
  - Best model      : /kaggle/working/best_absa_indobert
  - Test report     : /kaggle/working/test_classification_report.txt
{
  "best_config": {
    "name": "c4_p1_cw_192_bs8",
    "model_name": "indobenchmark/indobert-base-p1",
    "lr": 2e-05,
    "batch_size": 8,
    "max_epochs": 10,
    "max_len": 192,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "scheduler": "cosine",
    "optimizer": "adamw",
    "label_smoothing": 0.05,
    "freeze_layers": 0,
    "class_weight": true,
    "grad_accum": 2,
    "patience": 3,
    "seed": 42
  },
  "val_macro_f1": 0.7477314956208687,
  "test_metrics": {
    "macro_f1": 0.7454642699347086,
    "weighted_f1": 0.9289441710218542,
    "acc": 0.9283350666540607,
    "non_none_macro_f1": 0.6726495611752565,
    "aspect_detection_f1": 0.9177936558700134,
    "false_aspect_rate

## 13. Cara Inference Model Terbaik (referensi)

```python
def predict(texts, model, tok, max_len):
    model.eval()
    enc = tok([t.lower() for t in texts], truncation=True, max_length=max_len,
              padding=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    pred = logits.argmax(-1).cpu().numpy()  # shape (n, 7)
    out = []
    for row in pred:
        out.append({ASPECTS[i]: ID2LABEL[row[i]] for i in range(len(ASPECTS))})
    return out

# contoh:
# predict(['kamarnya bersih dan nyaman, pelayanan ramah'], best_model, best_tok, best_cfg['max_len'])
```
